**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Real-Time Signal Processing

The workshop [Intro to GPU Systems](../Intro_GPU/Intro_GPU.ipynb) promised: what changes when the signal *keeps coming* and every block has a **deadline**. Fixed-point arithmetic, block processing under a latency budget, and a simulated real-time pipeline with measured deadline misses — the glue between [DSP](./README.md), [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb), and [FPGA](../Intro_FPGA/README.md).

## 1. Pre-requisites

- [Filter Design](./Filter_Design.ipynb), [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) (block processing).
- [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — scheduling jitter is the enemy here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
import time
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Fixed-Point Arithmetic* (~40 min)
**Goal:** represent signals in Q-format; measure quantization noise; watch overflow bite.
**Builds on:** [Intro to C](../Intro_Programming/Intro_C.ipynb) (bits). &nbsp; **Feeds into:** Session 2 (latency budgets).

---

## 2. Numbers Without a Float Unit

💡 **Intuition.** Microcontrollers and [FPGA fabric](../Intro_FPGA/Intro_FPGA.ipynb) do integer math. **Q-format** fakes fractions with an implicit binary point: Q1.15 stores $x \in [-1, 1)$ as $\mathrm{round}(x \cdot 2^{15})$ in an int16. Each quantization adds ~uniform noise of variance $\Delta^2/12$ — *6 dB of SNR per bit* — and every multiply must be re-scaled (>> 15) or the binary point drifts. The two failure modes to respect: **quantization noise** (graceful, hissy) and **overflow** (catastrophic, wrap-around).

In [ ]:

# YOUR CODE HERE


In [ ]:
# A Q15 FIR filter, and the overflow trap
# CORRECT: accumulate in int32 (headroom!), shift back once
# WRONG: no headroom — products wrap in int16

# YOUR CODE HERE


---
### 🕐 Session 2 of 3 — *Latency Budgets & Block Processing* (~35 min)
**Goal:** count the milliseconds: block size sets the latency floor; compute must fit inside it.
**Builds on:** Session 1; [OS workshop](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb). &nbsp; **Feeds into:** Session 3 (a real-time pipeline).

---

## 3. The Budget

💡 **Intuition.** A real-time system processes block $k$ while block $k{+}1$ records. Two laws follow. **Latency floor:** you can't output before a block fills — latency ≥ one block (plus compute, plus output buffering); small blocks = low latency. **Throughput wall:** compute per block must finish in under one block-duration — or you fall behind *forever*. Small blocks also mean more per-block overhead ([OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) syscalls, scheduling), so the budget squeezes from both sides. Every audio interface's 'buffer size' knob is exactly this dial.

In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 3 of 3 — *A Real-Time Pipeline, Simulated & Measured* (~40 min)
**Goal:** producer/consumer with deadlines; measure misses and jitter like an engineer.
**Builds on:** Session 2.

---

## 4. The Pipeline

Architecture (the [OS workshop's](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) producer/consumer, with a clock): an acquisition thread produces blocks on schedule; a processing thread must consume + filter each block before the next arrives. We measure the **headroom histogram** — the engineer's dashboard for 'will this survive a bad scheduling day?'

In [ ]:

# YOUR CODE HERE


## 5. Conclusion

Six dB per bit, headroom before shifting, latency ≥ one block, compute < one block-duration, and always look at the *worst-case* headroom, not the average. When the budget can't be met on a CPU, you now know both escape hatches: [GPU batching](../Intro_GPU/README.md) (throughput, at latency cost) and [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) (deterministic latency, at effort cost).

---
## Where next

- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — the same Q15 FIR as literal hardware.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — MHz-rate streams where these budgets get serious.
- [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — the scheduler that owns your jitter.